# 5장. LangChain 기본

이 장은 PDF 교재의 `5. 랭체인(LangChain)`과 `LLM/llm.ipynb`, `LLM/llm2.ipynb`의 LangChain 관련 실습을 통합한 장입니다.

## 이 장에서 다루는 내용

1. LangChain이란 무엇인가
2. LangChain 주요 컴포넌트
3. LLM Wrappers
4. Prompt Templates
5. Memory
6. Chains
7. Agents
8. LangChain vs LlamaIndex
9. LCEL 기본 구조
10. `RunnablePassthrough`
11. `StrOutputParser`
12. `Ollama`와 `ChatOllama`
13. `HumanMessage`, `AIMessage`
14. 대화 이력 관리
15. `SQLChatMessageHistory`
16. 간단한 Retriever + Prompt + LLM 체인

LangChain은 이후 RAG, Gradio 챗봇, DB 챗봇, 자기소개서 생성 실습의 중심 뼈대입니다.


## 5.1 LangChain이란?

LangChain은 LLM 애플리케이션을 만들기 위한 프레임워크입니다.

단순히 LLM에 프롬프트를 보내는 것만으로는 실제 서비스를 만들기 어렵습니다. 실제 애플리케이션은 다음 기능을 함께 필요로 합니다.

- 프롬프트 템플릿 관리
- 다양한 LLM 연결
- 사용자 대화 이력 관리
- 외부 문서 검색
- 데이터베이스 연결
- API와 도구 호출
- 여러 단계를 연결한 워크플로 구성

LangChain은 이런 구성요소를 모듈로 제공하고, 서로 연결해 LLM 기반 애플리케이션을 만들 수 있게 해줍니다.


## 5.2 PDF 교재의 LangChain 주요 컴포넌트

PDF 5.1은 LangChain의 주요 컴포넌트를 다음처럼 정리합니다.

| 컴포넌트 | 설명 |
|---|---|
| LLM Wrappers | OpenAI, Hugging Face, Cohere, Ollama 등 다양한 LLM과 통합합니다. |
| Prompt Templates | LLM 프롬프트를 체계적으로 구성하고 관리합니다. |
| Memory | 대화 기록과 상태를 관리해 컨텍스트를 유지합니다. |
| Chains | 프롬프트, LLM, API 호출 등 여러 작업을 연결해 워크플로를 구성합니다. |
| Agents | 외부 도구, API, 데이터베이스와 상호작용하며 문제를 해결합니다. |

이 교재의 실습에서 LangChain은 다음 방식으로 반복 사용됩니다.

- Ollama 모델 호출
- Retriever와 LLM 연결
- RAG Chain 구성
- Gradio 챗봇의 대화 이력 처리
- SQL 대화 이력 저장
- DB 질의 챗봇
- 자기소개서 생성 Chain


## 5.3 LangChain의 기타 특징

PDF 교재는 LangChain의 기타 특징을 다음처럼 설명합니다.

| 특징 | 설명 |
|---|---|
| 다양한 LLM 지원 | GPT 계열, Hugging Face 모델, Claude, Ollama 등과 연결할 수 있습니다. |
| 외부 데이터 통합 | SQL, MongoDB, PDF, CSV, XLS 등 다양한 자료와 연결할 수 있습니다. |
| 확장성과 유연성 | 단순 호출부터 복잡한 파이프라인까지 다양한 요구에 맞게 구성할 수 있습니다. |

LangChain의 핵심은 `LLM 하나를 호출하는 코드`를 넘어 `LLM을 포함한 전체 작업 흐름`을 만드는 데 있습니다.


## 5.4 LangChain 설치

원본 노트북에 등장한 LangChain 관련 패키지는 다음과 같습니다.

- `langchain`
- `langchain-community`
- `langchain-core`
- `langchain-text-splitters`
- `faiss-cpu`
- `sentence-transformers`
- `chromadb`

LangChain은 버전 변화가 빠른 라이브러리입니다. 예전 코드에서는 `langchain.llms` 같은 경로를 쓰지만, 현재 원본 노트북은 주로 `langchain_community`, `langchain_core` 패키지 경로를 사용합니다.


In [ ]:
# LangChain 기본 실습용 설치 예시입니다.
# 필요한 경우 주석을 해제하고 실행하세요.

# %pip install langchain langchain-community langchain-core langchain-text-splitters
# %pip install faiss-cpu sentence-transformers chromadb


## 5.5 LangChain에서 Ollama 호출

PDF 5.2와 `llm.ipynb`는 Ollama 모델을 LangChain에서 호출하는 가장 기본적인 예제를 보여줍니다.

핵심 코드:

```python
from langchain_community.llms import Ollama
llm = Ollama(model="gemma2")
llm.invoke("안녕, 간단하게 소개해줘.")
```

`invoke()`는 LangChain 객체를 실행하는 표준 메서드입니다.

- LLM 객체에 문자열을 넣으면 답변 문자열을 반환합니다.
- Chain 객체에 질문을 넣으면 여러 단계가 순서대로 실행됩니다.
- Retriever 객체에 질문을 넣으면 관련 문서를 반환합니다.


In [ ]:
# LangChain의 Ollama LLM 래퍼를 불러옵니다.
from langchain_community.llms import Ollama

# 로컬 Ollama의 gemma2 모델을 연결합니다.
llm = Ollama(model="gemma2")

# invoke() 메서드로 모델에 프롬프트를 전달합니다.
llm.invoke("안녕, 간단하게 소개해줘.")


## 5.6 Prompt Template

Prompt Template은 프롬프트의 틀을 미리 만들고, 실행 시 변수만 바꿔 넣는 방식입니다.

예를 들어 RAG에서는 다음과 같은 프롬프트가 자주 사용됩니다.

```text
당신은 질문에 답변하는 AI 어시스턴트입니다.
제공된 context만을 바탕으로 질문에 답변하세요.

[Context]
{context}

[Question]
{question}
```

여기서 `{context}`와 `{question}`은 나중에 실제 값으로 채워지는 자리입니다.

LangChain에서는 `ChatPromptTemplate.from_template()` 또는 `ChatPromptTemplate.from_messages()`를 사용합니다.


In [ ]:
# ChatPromptTemplate은 채팅 모델에 보낼 프롬프트를 구성합니다.
from langchain_core.prompts import ChatPromptTemplate

# 문자열 템플릿을 정의합니다.
template = """
당신은 질문에 답변하는 AI 어시스턴트입니다.
아래 context만 사용해서 질문에 답하세요.

[Context]
{context}

[Question]
{question}
"""

# 문자열 템플릿을 LangChain 프롬프트 객체로 변환합니다.
prompt = ChatPromptTemplate.from_template(template)

# 프롬프트에 값을 채워 메시지를 만들어 봅니다.
prompt.format_messages(context="서울은 대한민국의 수도입니다.", question="대한민국의 수도는?")


## 5.7 LCEL, LangChain Expression Language

LCEL은 LangChain에서 여러 단계를 파이프처럼 연결하는 방식입니다.

대표 구조:

```python
chain = prompt | llm | StrOutputParser()
```

이 코드는 다음 흐름을 의미합니다.

```text
입력값 -> prompt -> llm -> output parser -> 최종 문자열
```

RAG에서는 다음처럼 더 복잡한 구조가 됩니다.

```python
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
```

이 구조는 `llm.ipynb`와 `llm2.ipynb`에서 반복해서 사용됩니다.


In [ ]:
# StrOutputParser는 LLM 응답에서 문자열만 깔끔하게 꺼내는 파서입니다.
from langchain_core.output_parsers import StrOutputParser

# 간단한 LCEL 체인입니다.
simple_chain = prompt | llm | StrOutputParser()

# context와 question 값을 넣어 체인을 실행합니다.
simple_chain.invoke({
    "context": "서울은 대한민국의 수도입니다.",
    "question": "대한민국의 수도는 어디야?"
})


## 5.8 RunnablePassthrough

`RunnablePassthrough`는 입력값을 변경하지 않고 다음 단계로 그대로 전달합니다.

RAG 체인에서 자주 쓰이는 구조:

```python
{"context": retriever, "question": RunnablePassthrough()}
```

이 구조의 의미는 다음과 같습니다.

| 키 | 값 | 의미 |
|---|---|---|
| `context` | `retriever` | 사용자 질문으로 관련 문서를 검색합니다. |
| `question` | `RunnablePassthrough()` | 사용자 질문 원문을 그대로 프롬프트에 전달합니다. |

즉 사용자가 입력한 질문 하나가 두 갈래로 나뉩니다.

1. Retriever로 들어가 관련 문서를 찾습니다.
2. 질문 자체는 그대로 프롬프트의 `{question}`에 들어갑니다.


In [ ]:
# RunnablePassthrough를 불러옵니다.
from langchain_core.runnables import RunnablePassthrough

# 입력값을 그대로 반환하는지 확인하는 간단한 예시입니다.
passthrough = RunnablePassthrough()
passthrough.invoke("이 문장은 그대로 전달됩니다.")


## 5.9 간단한 VectorStore + Retriever + Chain 예제

`llm.ipynb`에는 짧은 문장 네 개를 FAISS 벡터스토어에 넣고 질문에 답하는 간단한 예제가 있습니다.

문서 내용:

- 영준은 랭체인 주식회사에서 근무를 하였습니다.
- 설현은 테디와 같은 회사에서 근무하였습니다.
- 영준의 직업은 개발자입니다.
- 설현의 직업은 디자이너입니다.

이 예제는 RAG의 축소판입니다.

```text
텍스트 -> 임베딩 -> FAISS 저장 -> Retriever -> Prompt -> Ollama -> 답변
```


In [ ]:
# FAISS는 벡터 검색 저장소입니다.
# HuggingFaceEmbeddings는 텍스트를 벡터로 바꾸는 임베딩 모델입니다.
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

# 짧은 문장 목록을 벡터스토어로 만듭니다.
vectorstore = FAISS.from_texts(
    [
        "영준은 랭체인 주식회사에서 근무를 하였습니다.",
        "설현은 테디와 같은 회사에서 근무하였습니다.",
        "영준의 직업은 개발자입니다.",
        "설현의 직업은 디자이너입니다.",
    ],
    embedding=HuggingFaceEmbeddings(model_name='jhgan/ko-sroberta-multitask'),
)

# 벡터스토어를 Retriever로 변환합니다.
retriever = vectorstore.as_retriever()


In [ ]:
# 검색된 context만 기반으로 답변하게 하는 프롬프트입니다.
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""

# 프롬프트 객체를 생성합니다.
prompt = ChatPromptTemplate.from_template(template)

# Ollama 모델을 지정합니다.
model = Ollama(model="gemma2")

# Retriever, Prompt, LLM, Parser를 연결합니다.
retrieval_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)


In [ ]:
# 벡터스토어에 있는 정보로 답할 수 있는 질문입니다.
retrieval_chain.invoke("설현의 직업은?")


In [ ]:
# 영준이 근무한 곳을 검색 문맥에서 찾아 답합니다.
retrieval_chain.invoke("영준이 근무한 곳은?")


In [ ]:
# 검색 문맥에 없는 일반 추천 질문입니다.
# 이 경우 모델 자체의 생성 능력이 섞일 수 있으므로 RAG의 통제 필요성을 보여줍니다.
retrieval_chain.invoke("영준이 개발자라면 유망한 SW 기술 추천해줘. 한글로 설명해줘")


## 5.10 Memory와 대화 이력

PDF 5.1에서 Memory는 대화 기록과 상태를 관리해 컨텍스트를 유지하는 컴포넌트입니다.

일반 LLM 호출은 기본적으로 이전 대화를 기억하지 않습니다. 이전 대화를 기억하게 하려면 다음 중 하나가 필요합니다.

1. 이전 대화 내용을 매번 프롬프트에 같이 넣습니다.
2. LangChain Memory 또는 Message History를 사용합니다.
3. 데이터베이스에 대화 이력을 저장하고 다시 불러옵니다.

`llm.ipynb`에는 `SQLChatMessageHistory`를 사용해 SQLite DB에 대화 이력을 저장하는 예제가 있습니다.


In [ ]:
# SQLChatMessageHistory는 SQL DB에 대화 메시지를 저장합니다.
from langchain_community.chat_message_histories import SQLChatMessageHistory

# SQLite 파일 DB에 대화 이력을 저장합니다.
chat_message_history = SQLChatMessageHistory(
    session_id="sql_chat_history",
    connection="sqlite:///C:/work/LLM/history/chat_history.db"
)


In [ ]:
# 첫 번째 사용자 메시지를 대화 이력에 추가합니다.
chat_message_history.add_user_message(
    "안녕. 나는 영준이야. 직업은 웹프로그래머이고 만나서 반가워!"
)


In [ ]:
# 두 번째 사용자 메시지를 대화 이력에 추가합니다.
chat_message_history.add_user_message(
    "요즘 날씨가 추운데 건강 조심하고 즐거운 하루 보내!"
)


In [ ]:
# 저장된 메시지 목록을 확인합니다.
chat_message_history.messages


## 5.11 ChatOllama와 대화 기록 처리

`llm2.ipynb`의 Gradio 챗봇 예제는 `ChatOllama`, `HumanMessage`, `AIMessage`를 사용합니다.

대화 기록 처리 흐름:

1. Gradio가 이전 대화 기록 `history`를 전달합니다.
2. 각 `(human, ai)` 쌍을 LangChain 메시지 객체로 바꿉니다.
3. 현재 사용자 메시지를 추가합니다.
4. `model.invoke(chat_history)`로 전체 대화 문맥을 모델에 보냅니다.
5. 모델 응답의 `content`를 반환합니다.


In [ ]:
# ChatOllama와 메시지 객체를 불러옵니다.
from langchain_community.chat_models import ChatOllama
from langchain_core.messages import HumanMessage, AIMessage

# ChatOllama 모델을 초기화합니다.
chat_model = ChatOllama(model="gemma4:e2b", temperature=0.7, verbose=False)

# Gradio ChatInterface에서 사용할 수 있는 chat 함수 형태입니다.
def chat(message, history):
    # 이전 대화 기록을 LangChain 메시지 형식으로 변환합니다.
    chat_history = []
    for human, ai in history:
        chat_history.append(HumanMessage(content=human))
        chat_history.append(AIMessage(content=ai))

    # 현재 사용자 메시지를 추가합니다.
    chat_history.append(HumanMessage(content=message))

    # 모델을 호출합니다.
    response = chat_model.invoke(chat_history)

    # 응답 텍스트만 반환합니다.
    return response.content


## 5.12 LangChain vs LlamaIndex

PDF 5.3은 LangChain과 LlamaIndex를 비교합니다.

| 항목 | LangChain | LlamaIndex |
|---|---|---|
| 주요 초점 | 워크플로우 오케스트레이션 및 에이전트 | 데이터 인덱싱 및 RAG |
| 기능/강점 | 체인, 에이전트, 메모리, 도구 연결, 복잡한 다단계 워크플로우 | 대용량 데이터 인덱싱, 검색 품질, RAG 파이프라인 최적화 |
| 주요 구성요소 | 체인, 에이전트, 프롬프트 템플릿, 메모리, 도구 | 데이터 커넥터, 인덱싱 기능, 쿼리 엔진, 응답 합성 |
| 적합 사례 | 복잡한 챗봇, API/외부 서비스 연동, 다단계 AI 에이전트 | 기업 문서 검색, Q/A 시스템, 대용량 의미 검색 |
| 학습 곡선 | 유연하고 사용자 정의가 많아 상대적으로 가파름 | RAG 특화라 비교적 빠르게 시작 가능 |

이 교재는 LangChain을 중심으로 진행합니다. 이유는 원본 노트북의 대부분 실습이 LangChain 기반이기 때문입니다.


## 5.13 LangChain 코드 읽는 법

원본 노트북에는 다음 패턴이 자주 등장합니다.

### 1. 모듈 import

```python
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
```

### 2. 모델 생성

```python
llm = Ollama(model="gemma2")
```

### 3. 프롬프트 생성

```python
prompt = ChatPromptTemplate.from_template(template)
```

### 4. 체인 연결

```python
chain = prompt | llm | StrOutputParser()
```

### 5. 실행

```python
chain.invoke(input_data)
```

이 흐름만 이해하면 RAG, DB 챗봇, 자기소개서 생성 실습의 코드 구조도 훨씬 쉽게 읽을 수 있습니다.


## 5.14 자주 발생하는 오류와 해결 방법

### 1. `ModuleNotFoundError: langchain_community`

해결:

```python
%pip install langchain-community
```

설치 후 커널을 재시작합니다.

### 2. Ollama 연결 실패

해결:

- Ollama 앱 실행
- `ollama list` 확인
- 모델 설치 여부 확인
- `base_url="http://localhost:11434"` 확인

### 3. 임베딩 모델 다운로드 지연

처음 `HuggingFaceEmbeddings`를 사용할 때 모델을 다운로드하므로 시간이 걸릴 수 있습니다.

### 4. FAISS 설치 오류

Windows 환경에서는 보통 다음 패키지를 사용합니다.

```python
%pip install faiss-cpu
```

### 5. 대화 이력 DB 경로 오류

`sqlite:///C:/work/LLM/history/chat_history.db` 경로의 폴더가 없으면 오류가 날 수 있습니다. `C:/work/LLM/history` 폴더가 있는지 확인합니다.


## 5.15 이 장의 핵심 정리

이 장에서 배운 내용을 정리하면 다음과 같습니다.

- LangChain은 LLM 애플리케이션을 만들기 위한 프레임워크입니다.
- 주요 컴포넌트는 LLM Wrappers, Prompt Templates, Memory, Chains, Agents입니다.
- `Ollama`는 LangChain에서 로컬 LLM을 호출하는 래퍼입니다.
- `ChatOllama`는 메시지 기반 대화형 모델 호출에 사용합니다.
- `ChatPromptTemplate`은 프롬프트를 구조화합니다.
- `StrOutputParser`는 LLM 응답을 문자열로 변환합니다.
- `RunnablePassthrough`는 입력 질문을 그대로 다음 단계에 전달합니다.
- LCEL은 `prompt | llm | parser`처럼 체인을 직관적으로 연결합니다.
- Retriever, Prompt, LLM을 연결하면 RAG의 기본 구조가 됩니다.
- Memory는 대화 이력을 저장하고 문맥을 유지하는 데 필요합니다.
- `SQLChatMessageHistory`로 SQLite나 MySQL에 대화 기록을 저장할 수 있습니다.

다음 장에서는 Gradio를 사용해 LangChain/Ollama 챗봇을 웹 UI로 만드는 방법을 다룹니다.
